# Model Inference: Comparison, Refit, and Kaggle Submission

This notebook pulls results from all trained architectures from MLflow, selects the best submission-capable model, refits it on the full train.csv, and generates the final Kaggle submission.

In [1]:
import mlflow
import dagshub
import pandas as pd
import sys

sys.path.append("..")
from src.data.load_data import load_raw_data

dagshub.init(repo_owner="LukaBatilashvili07", repo_name="walmart-sales-forecasting", mlflow=True)
client = mlflow.MlflowClient()

Accessing as LukaBatilashvili07

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## 1. Compare Validation WMAE Across All Models

In [3]:
EXPERIMENTS = {
    "XGBoost": ("XGBoost_Training", "XGBoost_Best", "final_valid_wmae"),
    "LightGBM": ("LightGBM_Training", "LightGBM_Best", "final_valid_wmae"),
    "DLinear": ("DLinear_Training", "DLinear_TargetOnly_Kernel25", "valid_wmae"),
}

results = []
for arch, (exp_name, run_name, metric) in EXPERIMENTS.items():
    exp = client.get_experiment_by_name(exp_name)
    run = client.search_runs(exp.experiment_id, f"tags.mlflow.runName = '{run_name}'")[0]
    results.append({"architecture": arch, "wmae": run.data.metrics[metric]})

comparison_df = pd.DataFrame(results).sort_values("wmae")
print(comparison_df)

best_arch = comparison_df.iloc[0]["architecture"]
print("\nBest model:", best_arch)

  architecture         wmae
2      DLinear  2601.292163
0      XGBoost  4638.494814
1     LightGBM  4891.093745

Best model: DLinear


## 2. Load the Full-History Version of the Winning Pipeline

Each model notebook already refit its best model on the full train.csv and registered it. We just load that version here, no retraining.

In [12]:
REGISTRY_NAMES = {
    "XGBoost": "Walmart_XGBoost_Pipeline",
    "LightGBM": "Walmart_LightGBM_Pipeline",
    "DLinear": "Walmart_DLinear_Pipeline",
}

In [13]:
def get_latest_version(client, model_name):
    versions = client.search_model_versions(f"name='{model_name}'")
    if not versions:
        return None
    latest = max(versions, key=lambda v: int(v.version))
    return latest.version


ranked = comparison_df.sort_values("wmae")["architecture"].tolist()

model_name = None
version = None
for arch in ranked:
    candidate_name = REGISTRY_NAMES[arch]
    candidate_version = get_latest_version(client, candidate_name)
    if candidate_version is not None:
        best_arch = arch
        model_name = candidate_name
        version = candidate_version
        break

if model_name is None:
    raise RuntimeError("No registered model found for any candidate architecture.")

print(f"Using best AVAILABLE model: {best_arch} ({model_name} v{version})")
if best_arch != ranked[0]:
    print(f"NOTE: {ranked[0]} scored better but has no registered full-history pipeline yet.")

final_pipeline = mlflow.pyfunc.load_model(f"models:/{model_name}/{version}")

Using best AVAILABLE model: XGBoost (Walmart_XGBoost_Pipeline v2)
NOTE: DLinear scored better but has no registered full-history pipeline yet.


## 3. Predict on the Raw Kaggle Test Set

In [15]:
train_raw, test_raw, stores, features = load_raw_data("../data/raw")

test_preds = final_pipeline.predict(test_raw)
print(test_preds[:10])

[37490.902 16816.447 21373.172 28265.648 27287.223 30050.262 32704.934
 38583.74  22029.943 19568.887]


## 4. Build and Save the Submission File

In [18]:
submission = pd.DataFrame({
    "Id": test_raw["Store"].astype(str) + "_" + test_raw["Dept"].astype(str) + "_" + test_raw["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_preds,
})
submission["Weekly_Sales"] = submission["Weekly_Sales"].clip(lower=0)

submission.to_csv("../submission.csv", index=False)
print(submission.shape)
submission.head()

(115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,37490.902344
1,1_1_2012-11-09,16816.447266
2,1_1_2012-11-16,21373.171875
3,1_1_2012-11-23,28265.648438
4,1_1_2012-11-30,27287.222656


In [19]:
print(submission["Weekly_Sales"].describe())
print((submission["Weekly_Sales"] < 0).sum(), "negative predictions")

count    115064.000000
mean      15145.720703
std       20672.263672
min           0.000000
25%        3319.899841
50%        6914.901855
75%       18204.960449
max      371232.156250
Name: Weekly_Sales, dtype: float64
0 negative predictions
